# 01. EDA & Data Preprocessing Pipeline (Version 1)
**Phụ trách:** Người 1 (Data & Pipeline Lead)

Notebook này kiểm tra dữ liệu thô, chạy pipeline V1 tạo 3 file `.parquet` (`train_v1`, `val_v1`, `test_v1`) và tính thử điểm Baseline RMSLE.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.metrics import compute_rmsle
from src.preprocess import build_v1_dataset

In [ ]:
# Chạy pipeline V1 nếu chưa chạy từ terminal
build_v1_dataset(raw_dir='../data/raw', processed_dir='../data/processed')

In [ ]:
# Đánh giá điểm Baseline RMSLE trên tập Validation 16 ngày
train_v1 = pd.read_parquet('../data/processed/train_v1.parquet')
val_v1 = pd.read_parquet('../data/processed/val_v1.parquet')

baseline_map = train_v1.groupby(['store_nbr', 'family'], observed=True)['sales'].mean().reset_index()
baseline_map.rename(columns={'sales': 'pred_sales'}, inplace=True)

val_eval = val_v1.merge(baseline_map, on=['store_nbr', 'family'], how='left')
score = compute_rmsle(val_eval['sales'], val_eval['pred_sales'])
print(f'Baseline Validation RMSLE: {score:.5f}')